# MODEL TRAINING AND COMPARISON

Day-ahead (48 half-hours ahead) forecasting of GB national demand with LSTM, XGBoost and
multiple linear regression (MLR), each trained on the full feature set and, for LSTM and
XGBoost, on a feature subset chosen by binary MRFO. Models are scored on the whole test set
(last 20% of the data), against MLR and a seasonal-naive baseline, and compared with Wilcoxon
and Diebold-Mariano tests.

## SETUP AND DATA LOADING

In [ ]:
import os
import random
from datetime import datetime

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import xgboost as xgb
import shap
import mlflow
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from statsmodels.tsa.api import VAR

from energy_forecast.evaluation.metrics import SCALAR_METRICS, TRAIN_METRICS, evaluate
from energy_forecast.evaluation.significance import diebold_mariano_test
from energy_forecast.features.mrfo import (
    manta_ray_foraging_optimization,
    random_search,
    xgboost_fitness_function,
    lstm_fitness_function,
)
from energy_forecast.models.lstm_model import (
    LSTM,
    prepare_lstm_data,
    train_model,
    train_with_early_stopping,
    predict_train_set,
    predict_test_set,
)
from energy_forecast.models.tuning import tune

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Separate experiment for the day-ahead setup, so runs from the earlier one-step setup don't mix in
MLFLOW_EXPERIMENT = "energy-demand-forecasting-day-ahead"
mlflow.set_experiment(MLFLOW_EXPERIMENT)

# Tags every run from this execution of the notebook, so the cross-check only shows these runs
RUN_GROUP = datetime.now().strftime("%Y%m%d-%H%M%S")

In [ ]:
# Fixed LSTM settings; the remaining hyperparameters are tuned below
seq_length = 48  # input window: one day of half-hours
output_size = 1
criterion = nn.MSELoss()

# Tuning: every model gets the same number of random configurations, scored on the validation set
TUNING_TRIALS = 10
LSTM_MAX_EPOCHS = 30  # upper bound; early stopping on validation loss decides the actual count
LSTM_PATIENCE = 3

In [ ]:
if os.getcwd().endswith("notebooks"):
    os.chdir("..")

In [ ]:
full = pd.read_csv("data/processed/model_ready_features_day_ahead.csv", index_col=0, parse_dates=True)

### Evaluation

Every model is scored with `evaluate()` from `src/energy_forecast/evaluation/metrics.py`, on
the **entire test set** (last 20% of the data, Feb 2021 – Feb 2024), not just its first week.
Besides overall MAE and RMSE, two relative metrics common in the load-forecasting literature
are reported: MAPE (mean absolute percentage error) and nRMSE (RMSE divided by mean demand).
RMSE is also computed per calendar week, to show how stable each model is across seasons,
holidays and years. The Diebold-Mariano test used later lives in
`src/energy_forecast/evaluation/significance.py`; both are covered by `tests/test_evaluation.py`.

## FEATURE SELECTION (MRFO)

MRFO feature selection only sees the training portion (first 80%) of the data. Inside each
fitness evaluation, that portion is split again into an inner training set (first 80% of it)
and a validation set (last 20% of it), and candidate feature subsets are scored on the
validation set. The test set (last 20%) is never used during selection, so it stays unseen
until final evaluation. The selected features are then trained on the entire training set,
as for the other data sets.

In [ ]:
# Mirrors the splits used below: outer 80/20 train/test, then MRFO's inner 80/20 train/validation
train_end = int(0.8 * len(full))
val_start = int(0.8 * train_end)

plt.plot(full.index[:val_start], full['nd'].iloc[:val_start], color='cyan', label='MRFO training set')
plt.plot(full.index[val_start:train_end], full['nd'].iloc[val_start:train_end], color='orange', label='MRFO validation set')
plt.plot(full.index[train_end:], full['nd'].iloc[train_end:], color='purple', label='Test set (unseen by MRFO)')

plt.ylabel('National Demand (nd)')
plt.title('Train / Validation / Test Split')
plt.legend()
plt.show()

### Binary MRFO

Feature subsets are searched with binary Manta Ray Foraging Optimization (Zhao, Zhang & Wang,
2020). Each manta ray has a continuous position in [0, 1] per feature, and a feature is selected
when its coordinate is above 0.5. Every iteration, each ray does either **chain foraging** (moving
towards the ray in front and the best solution) or **cyclone foraging** (spiralling around a
random point early on for exploration, and around the best solution later for exploitation). All
rays then do **somersault foraging** around the best solution.

Subsets are scored by an objective to minimise:

$$\text{objective} = 0.99 \cdot \frac{\text{validation MSE}}{\operatorname{Var}(y_{\text{val}})} + 0.01 \cdot \frac{n_{\text{selected}}}{n_{\text{features}}}$$

The small feature-count penalty makes MRFO drop features that don't improve validation error.
Scores are cached per subset (each subset is trained once), and the LSTM fitness is seeded, so
the same subset always gets the same score.

In [ ]:
X = full.drop(columns=['nd'])
y = full['nd']

# Only the training portion goes into MRFO; the fitness functions split off their own
# validation set from it, so the test set is never used to choose features
X_mrfo, _, y_mrfo, _ = train_test_split(X, y, test_size=0.2, shuffle=False)

# Same budget for both fitness functions, so differences in the selected subsets come from
# the model rather than the search effort (10 x 10 keeps the LSTM search affordable)
MRFO_ITERATIONS = 10
MRFO_RAYS = 10
# Weight of the validation error vs the feature-count penalty (see the check below)
MRFO_ALPHA = 0.99

mrfo_xgb = manta_ray_foraging_optimization(
    X_mrfo, y_mrfo, fitness_function=xgboost_fitness_function,
    num_iterations=MRFO_ITERATIONS, num_manta_rays=MRFO_RAYS, alpha=MRFO_ALPHA, random_state=42,
)
selected_columns_xgb = X.columns[np.where(mrfo_xgb.best_mask == 1)[0]]
print("XGBoost-MRFO selected:", list(selected_columns_xgb))

mrfo_lstm = manta_ray_foraging_optimization(
    X_mrfo, y_mrfo, fitness_function=lstm_fitness_function,
    num_iterations=MRFO_ITERATIONS, num_manta_rays=MRFO_RAYS, alpha=MRFO_ALPHA, random_state=42,
    num_epochs=5,
)
selected_columns_lstm = X.columns[np.where(mrfo_lstm.best_mask == 1)[0]]
print("LSTM-MRFO selected:", list(selected_columns_lstm))

#### Penalty check

The objective adds `(1 − alpha) · n_selected / n_features` to `alpha · normalised validation
error`. The penalty should only break ties between subsets with similar error. If it is a large
share of the best objective, MRFO may be dropping features that genuinely help: raise
`MRFO_ALPHA` (e.g. to 0.999) and re-run the searches.

In [ ]:
for name, result in [("XGBoost-MRFO", mrfo_xgb), ("LSTM-MRFO", mrfo_lstm)]:
    share = result.best_penalty_term / result.best_objective
    print(
        f"{name}: error term {result.best_error_term:.5f}, penalty term {result.best_penalty_term:.5f} "
        f"({share:.0%} of the objective, {result.best_mask.sum()} features)"
    )

In [ ]:
xgb_mrfo = full[list(selected_columns_xgb) + ['nd']]
lstm_mrfo = full[list(selected_columns_lstm) + ['nd']]

### Convergence and random-search baseline

To check that MRFO is actually optimising, the XGBoost-fitness search is compared with random
search given the same number of objective evaluations. (The LSTM search is too expensive to
repeat as a random baseline, so only its convergence curve is shown.)

In [ ]:
random_xgb = random_search(
    X_mrfo, y_mrfo, fitness_function=xgboost_fitness_function,
    num_evaluations=len(mrfo_xgb.history), alpha=MRFO_ALPHA, random_state=42,
)

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].plot(mrfo_xgb.history, label=f"MRFO ({mrfo_xgb.n_unique_evaluations} unique subsets)")
axs[0].plot(random_xgb.history, label=f"Random search ({random_xgb.n_unique_evaluations} unique subsets)")
axs[0].set_title("XGBoost fitness: best objective so far")
axs[0].set_xlabel("Objective evaluations")
axs[0].set_ylabel("Objective (lower is better)")
axs[0].legend()

axs[1].plot(mrfo_lstm.history, color="teal", label=f"MRFO ({mrfo_lstm.n_unique_evaluations} unique subsets)")
axs[1].set_title("LSTM fitness: best objective so far")
axs[1].set_xlabel("Objective evaluations")
axs[1].set_ylabel("Objective (lower is better)")
axs[1].legend()

plt.tight_layout()
plt.show()

## HYPERPARAMETER TUNING

XGBoost and the LSTM each get the **same budget**: `TUNING_TRIALS` random configurations from a
small grid, so neither architecture wins just because it was configured better. Each
configuration is trained on the first 80% of the training data and scored (RMSE) on the last
20%, the same validation split MRFO uses; the test set is never touched.

Early stopping on the validation set decides how many trees (XGBoost) or epochs (LSTM) to use.
The final models are then refit on the whole training set (first 80% of the data) with the best
configuration and that number of trees/epochs. Tuning uses the full feature set, and the tuned
settings are reused for the MRFO feature sets.

In [ ]:
X_tune_train, X_val, y_tune_train, y_val = train_test_split(X_mrfo, y_mrfo, test_size=0.2, shuffle=False)

XGB_SEARCH_SPACE = {
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.03, 0.05, 0.1],
    "min_child_weight": [1, 5, 10],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
}


def score_xgb(config):
    model = xgb.XGBRegressor(
        n_estimators=2000, early_stopping_rounds=50, n_jobs=-1, random_state=42, **config
    )
    model.fit(X_tune_train, y_tune_train, eval_set=[(X_val, y_val)], verbose=False)
    # predict() uses the best iteration found by early stopping
    val_rmse = np.sqrt(mean_squared_error(y_val, model.predict(X_val)))
    return {"score": val_rmse, "n_estimators": model.best_iteration + 1}


best_xgb, xgb_trials = tune(XGB_SEARCH_SPACE, score_xgb, n_trials=TUNING_TRIALS, random_state=42)
XGB_PARAMS = {
    **{name: best_xgb[name] for name in XGB_SEARCH_SPACE},
    "n_estimators": best_xgb["n_estimators"],
    "n_jobs": -1,
    "random_state": 42,
}
print("Best XGBoost settings:", XGB_PARAMS)
xgb_trials

In [ ]:
LSTM_SEARCH_SPACE = {
    "hidden_size": [32, 64, 128],
    "num_layers": [1, 2],
    "dropout": [0.1, 0.25],  # only applies between stacked layers
    "learning_rate": [3e-4, 1e-3, 3e-3],
    "batch_size": [256, 512],
}


def score_lstm(config):
    train_loader, val_loader, _, y_scaler, _, y_val_arr = prepare_lstm_data(
        X_tune_train, X_val, y_tune_train, y_val, seq_length, config["batch_size"]
    )
    torch.manual_seed(42)
    model = LSTM(
        X_tune_train.shape[1], config["hidden_size"], config["num_layers"], output_size,
        dropout=config["dropout"],
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    model, _, _, best_epoch = train_with_early_stopping(
        model, train_loader, val_loader, criterion, optimizer,
        max_epochs=LSTM_MAX_EPOCHS, patience=LSTM_PATIENCE,
    )
    val_rmse = predict_test_set(model, val_loader, y_val_arr, y_scaler)["rmse"]
    return {"score": val_rmse, "num_epochs": best_epoch}


best_lstm, lstm_trials = tune(LSTM_SEARCH_SPACE, score_lstm, n_trials=TUNING_TRIALS, random_state=42)
LSTM_PARAMS = {name: best_lstm[name] for name in LSTM_SEARCH_SPACE} | {"num_epochs": best_lstm["num_epochs"]}
print("Best LSTM settings:", LSTM_PARAMS)
lstm_trials

## MODELS

All models make **day-ahead direct forecasts**: to predict demand at time t they only use
demand observed up to t − 48 (the lag and rolling features built in notebook 2 are shifted by
48 half-hours). No predictions are fed back in as inputs. XGBoost and MLR see one row of
features per target; the LSTM sees a window of the 48 feature rows ending at the target, and
every demand-based value in that window is also at least 48 steps old. Weather and embedded
generation are observed values (the "perfect weather forecast" assumption).

XGBoost and the LSTM use the settings tuned above; MLR and the seasonal naive have none to tune.

### LSTM

In [ ]:
lstm_data_configs = {
    "full": full,
    "mrfo_lstm": lstm_mrfo,
}

results = {}

for config_name, data in lstm_data_configs.items():
    X = data.drop(columns=["nd"])
    y = data["nd"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    train_loader, test_loader, x_scaler, y_scaler, y_train_arr, y_test_arr = prepare_lstm_data(
        X_train, X_test, y_train, y_test, seq_length, LSTM_PARAMS["batch_size"]
    )

    with mlflow.start_run(run_name=f"lstm_{config_name}", tags={"run_group": RUN_GROUP}) as run:
        mlflow.log_params({
            "model_type": "LSTM",
            "feature_set": config_name,
            **LSTM_PARAMS,
            "seq_length": seq_length,
            "n_features": X_train.shape[1],
        })

        # Reseed so initial weights don't depend on MRFO or on the previous config having run
        torch.manual_seed(42)
        model = LSTM(
            X_train.shape[1], LSTM_PARAMS["hidden_size"], LSTM_PARAMS["num_layers"], output_size,
            dropout=LSTM_PARAMS["dropout"],
        ).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=LSTM_PARAMS["learning_rate"])

        # Refit on the whole training set for the number of epochs early stopping chose in tuning
        trained_model, loss = train_model(model, train_loader, criterion, optimizer, LSTM_PARAMS["num_epochs"])

        for epoch, epoch_loss in enumerate(loss):
            mlflow.log_metric("train_loss", epoch_loss, step=epoch)

        mlflow.pytorch.log_model(trained_model, name="model")

        results[config_name] = {
            "train_loader": train_loader,
            "test_loader": test_loader,
            "y_scaler": y_scaler,
            "y_train": y_train_arr,
            "y_test": y_test_arr,
            "train_index": X_train.index,
            "test_index": X_test.index,
            "loss": loss,
            "trained_model": trained_model,
            "mlflow_run_id": run.info.run_id,
        }

In [ ]:
# Training-set scores, to compare with the test scores (over/underfitting check)
for config_name, result in results.items():
    print(f"Training-set predictions for {config_name}")
    train_predictions = predict_train_set(result["trained_model"], result["train_loader"], result["y_scaler"])

    # The first training window targets row seq_length - 1, so the first 47 rows have no prediction
    offset = seq_length - 1
    train_metrics = evaluate(
        result["y_train"][offset:], train_predictions, result["train_index"][offset:], split="train"
    )
    result.update({k: train_metrics[k] for k in TRAIN_METRICS})

    with mlflow.start_run(run_id=result["mlflow_run_id"]):
        mlflow.log_metrics({k: result[k] for k in TRAIN_METRICS})

In [ ]:
for config_name, result in results.items():
    print(f"Test-set predictions for {config_name}")
    test_results = predict_test_set(
        result["trained_model"], result["test_loader"], result["y_test"], result["y_scaler"]
    )

    result["test_predictions"] = test_results["predictions"]
    result.update(evaluate(result["y_test"], result["test_predictions"], result["test_index"]))

    with mlflow.start_run(run_id=result["mlflow_run_id"]):
        mlflow.log_metrics({k: result[k] for k in SCALAR_METRICS})

### XGBoost

In [ ]:
xgb_data_configs = {
    "xgb_full": full,
    "xgb_mrfo": xgb_mrfo,  # XGBoost-fitness MRFO selection
}

xgb_results = {}

for config_name, data in xgb_data_configs.items():
    X = data.drop(columns=["nd"])
    y = data["nd"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    with mlflow.start_run(run_name=config_name, tags={"run_group": RUN_GROUP}):
        mlflow.log_params({"model_type": "XGBoost", "feature_set": config_name, "n_features": X_train.shape[1], **XGB_PARAMS})

        model = xgb.XGBRegressor(**XGB_PARAMS)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        metrics = evaluate(y_test, y_pred, X_test.index)
        train_metrics = evaluate(y_train, model.predict(X_train), X_train.index, split="train")

        mlflow.log_metrics({k: metrics[k] for k in SCALAR_METRICS})
        mlflow.log_metrics({k: train_metrics[k] for k in TRAIN_METRICS})

        mlflow.xgboost.log_model(model, name="model")

        xgb_results[config_name] = {
            "model": model, "X_test": X_test, "y_test": y_test, "y_pred": y_pred,
            **metrics,
            **{k: train_metrics[k] for k in TRAIN_METRICS},
        }

### Multiple Linear Regression

In [ ]:
X_mlr = full.drop(columns=['nd'])
y_mlr = full['nd']

X_train_mlr, X_test_mlr, y_train_mlr, y_test_mlr = train_test_split(X_mlr, y_mlr, test_size=0.2, shuffle=False)

with mlflow.start_run(run_name="mlr_full", tags={"run_group": RUN_GROUP}):
    mlflow.log_params({
        "model_type": "MLR",
        "feature_set": "full",
        "n_features": X_train_mlr.shape[1],
    })

    model_mlr = LinearRegression()
    model_mlr.fit(X_train_mlr, y_train_mlr)

    y_pred_mlr = model_mlr.predict(X_test_mlr)
    mlr_metrics = evaluate(y_test_mlr, y_pred_mlr, X_test_mlr.index)
    mlr_train_metrics = evaluate(y_train_mlr, model_mlr.predict(X_train_mlr), X_train_mlr.index, split="train")
    mlr_metrics |= {k: mlr_train_metrics[k] for k in TRAIN_METRICS}
    rmse_mlr, mae_mlr = mlr_metrics["rmse_test"], mlr_metrics["mae_test"]

    mlflow.log_metrics({k: mlr_metrics[k] for k in SCALAR_METRICS + TRAIN_METRICS})

    mlflow.sklearn.log_model(model_mlr, name="model")

print(f'RMSE/MAE (full test set): {rmse_mlr}, {mae_mlr}')

### Seasonal-naive baseline

Predicts each half-hour's demand as the value at the same time one week earlier
(`nd_lag_336`, available day-ahead). This is the standard reference in load forecasting:
any useful model should beat it.

In [ ]:
y_pred_naive = X_test_mlr["nd_lag_336"].to_numpy()
naive_metrics = evaluate(y_test_mlr, y_pred_naive, X_test_mlr.index)
# No fitting involved: the training-period score just shows the baseline's error level there
naive_train_metrics = evaluate(y_train_mlr, X_train_mlr["nd_lag_336"], X_train_mlr.index, split="train")
naive_metrics |= {k: naive_train_metrics[k] for k in TRAIN_METRICS}

with mlflow.start_run(run_name="seasonal_naive", tags={"run_group": RUN_GROUP}):
    mlflow.log_params({"model_type": "SeasonalNaive", "feature_set": "nd_lag_336"})
    mlflow.log_metrics({k: naive_metrics[k] for k in SCALAR_METRICS + TRAIN_METRICS})

print(f'RMSE/MAE (full test set): {naive_metrics["rmse_test"]}, {naive_metrics["mae_test"]}')

## COMPARISON

The comparison table reports test-set metrics for every model, plus training-set RMSE and nRMSE
as an over/underfitting check. `nrmse_test_to_train` well above 1 means a model fits the data it
was trained on much better than new data. The seasonal-naive baseline involves no fitting, so
its ratio shows how much of the gap comes from the test period simply being different.

In [ ]:
test_index = results["full"]["test_index"]
y_actual = results["full"]["y_test"].flatten()

# Every model uses the same chronological split, so all predictions share these timestamps
assert xgb_results["xgb_full"]["y_test"].index.equals(test_index)
assert y_test_mlr.index.equals(test_index)

predictions = {
    **{f"LSTM {name}": r["test_predictions"] for name, r in results.items()},
    **{f"XGBoost {name.removeprefix('xgb_')}": r["y_pred"] for name, r in xgb_results.items()},
    "MLR": y_pred_mlr,
    "Seasonal naive": y_pred_naive,
}

for n_steps, period in [(336, "week"), (48, "day")]:
    plt.figure(figsize=(14, 6))
    plt.plot(test_index[:n_steps], y_actual[:n_steps], color="black", linewidth=2, label="Actual")
    for name, preds in predictions.items():
        plt.plot(test_index[:n_steps], np.asarray(preds)[:n_steps], label=name)
    plt.title(f"Day-ahead forecasts — first {period} of test period")
    plt.xlabel("Date")
    plt.ylabel("Demand (MW)")
    plt.legend(loc="lower left")
    plt.show()

In [ ]:
all_results = {
    **{f"LSTM_{config_name}": result for config_name, result in results.items()},
    **xgb_results,
    "MLR": mlr_metrics,
    "Seasonal naive": naive_metrics,
}

# One column per model, one row per full calendar week of the test period
weekly_rmse = pd.DataFrame({name: r["weekly_rmse"] for name, r in all_results.items()})

comparison_df = pd.DataFrame({
    metric: {name: r[metric] for name, r in all_results.items()}
    for metric in SCALAR_METRICS + ["rmse_train", "nrmse_train"]
})
# Over/underfitting check. Uses nRMSE because demand was higher in the training years, so raw
# RMSE isn't directly comparable: well above 1 means the model fits the training data much
# better than it generalises; close to the seasonal-naive ratio means it generalises as well
# as a model that can't overfit at all
comparison_df["nrmse_test_to_train"] = comparison_df["nrmse_test"] / comparison_df["nrmse_train"]
comparison_df["weekly_rmse_median"] = weekly_rmse.median()
comparison_df["weekly_rmse_worst"] = weekly_rmse.max()

# Skill scores: 1 - RMSE_model / RMSE_baseline (> 0 means better than the baseline)
comparison_df["skill_vs_naive"] = 1 - comparison_df["rmse_test"] / naive_metrics["rmse_test"]
comparison_df["skill_vs_mlr"] = 1 - comparison_df["rmse_test"] / rmse_mlr
# Share of test weeks in which the model's weekly RMSE is below MLR's
comparison_df["weeks_beating_mlr"] = weekly_rmse.lt(weekly_rmse["MLR"], axis=0).mean()

comparison_df.sort_values("rmse_test")

In [ ]:
ax = weekly_rmse.plot(figsize=(14, 6))
ax.set_title("Weekly RMSE across the test period")
ax.set_xlabel("Week")
ax.set_ylabel("RMSE (MW)")
plt.show()

In [ ]:
runs_df = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT],
    filter_string=f"tags.run_group = '{RUN_GROUP}'",
)
runs_df[["tags.mlflow.runName", "params.feature_set", "metrics.rmse_test", "metrics.mae_test", "metrics.mape_test"]]

### Statistical tests: Wilcoxon and Diebold-Mariano

Each model pair is compared with two tests:

- **Wilcoxon signed-rank**: a simple, familiar non-parametric baseline, applied to paired
  squared errors so it tests which model is more accurate.
- **Diebold-Mariano (DM)**: the standard test in the forecasting literature. It compares the
  loss differential between two forecasts and uses a Newey-West-style long-run variance, so it
  can account for autocorrelation in forecast errors.

Both tests use squared-error loss, consistent with RMSE (the headline metric and the basis of
the skill score), the LSTM's MSE training loss and the MRFO objective.

`h=48` is used because every forecast is 48 steps (one day) ahead. Errors of h-step-ahead
forecasts are autocorrelated up to lag h − 1 by construction, so the long-run variance uses
47 lags (the default of `h - 1` in `diebold_mariano_test`).

#### Pairwise model comparisons

- **Baseline (seasonal naive vs XGBoost / LSTM):** the minimum bar for a useful forecast.
- **Baseline (MLR vs XGBoost / LSTM):** MLR uses the same features and the same test timestamps,
  but is linear and has no sequence memory. Beating it shows the value of nonlinearity (XGBoost)
  or temporal memory (LSTM) on identical inputs. The different feature handling (no scaling for
  MLR/XGBoost) does not affect the tests: they only compare errors in MW at matched timestamps.
- **Architecture (XGBoost vs LSTM):** both on the full feature set.
- **Feature selection:** whether MRFO improves each model over using all features, and whether
  MRFO's selection differs meaningfully depending on which model it was optimised for.

The tests use the errors over the entire test period (~53k half-hours). They are stored as
timestamp-indexed series and paired with an inner join, so every test compares the same
half-hours. With eight tests, p-values are Holm-corrected for multiple comparisons; significance
is judged on the corrected values. With this many points even small differences come out
significant, so read the p-values together with the skill score and `weeks_beating_mlr` in the
comparison table, which show how large and how consistent a difference is.

In [ ]:
errors = {
    "Seasonal naive": naive_metrics["errors"],
    "MLR": mlr_metrics["errors"],
    "XGBoost full": xgb_results["xgb_full"]["errors"],
    "XGBoost mrfo": xgb_results["xgb_mrfo"]["errors"],
    "LSTM full": results["full"]["errors"],
    "LSTM mrfo_lstm": results["mrfo_lstm"]["errors"],
}
n_test = len(errors["MLR"])

test_pairs = {
    "Baseline: seasonal naive vs XGBoost full": ("Seasonal naive", "XGBoost full"),
    "Baseline: seasonal naive vs LSTM full": ("Seasonal naive", "LSTM full"),
    "Baseline: MLR vs XGBoost full": ("MLR", "XGBoost full"),
    "Baseline: MLR vs LSTM full": ("MLR", "LSTM full"),
    "Architecture: XGBoost full vs LSTM full": ("XGBoost full", "LSTM full"),
    "XGBoost: full vs mrfo": ("XGBoost full", "XGBoost mrfo"),
    "LSTM: full vs mrfo_lstm": ("LSTM full", "LSTM mrfo_lstm"),
    "MRFO comparison: xgb_mrfo vs lstm_mrfo": ("XGBoost mrfo", "LSTM mrfo_lstm"),
}

rows = []
for label, (name_a, name_b) in test_pairs.items():
    errors_a, errors_b = errors[name_a].align(errors[name_b], join="inner")
    assert len(errors_a) == n_test, f"{label}: only {len(errors_a)} shared timestamps"

    # Both tests compare squared errors (tests accuracy, not whether errors differ in sign/bias)
    w_stat, w_p = wilcoxon(errors_a ** 2, errors_b ** 2)
    dm_stat, dm_p = diebold_mariano_test(errors_a.to_numpy(), errors_b.to_numpy(), h=48, power=2)
    rows.append({
        "comparison": label,
        "wilcoxon_stat": w_stat, "wilcoxon_p": w_p,
        "dm_stat": dm_stat, "dm_p": dm_p,
    })

tests_df = pd.DataFrame(rows).set_index("comparison")
tests_df["wilcoxon_p_holm"] = multipletests(tests_df["wilcoxon_p"], method="holm")[1]
tests_df["dm_p_holm"] = multipletests(tests_df["dm_p"], method="holm")[1]


def verdict(p_value, alpha=0.05):
    return "significant" if p_value < alpha else "not significant"


for label, row in tests_df.iterrows():
    print(label)
    print(f"  Wilcoxon: statistic={row.wilcoxon_stat:.2f}, p={row.wilcoxon_p:.4f}, p_holm={row.wilcoxon_p_holm:.4f} ({verdict(row.wilcoxon_p_holm)})")
    print(f"  DM:       statistic={row.dm_stat:.2f}, p={row.dm_p:.4f}, p_holm={row.dm_p_holm:.4f} ({verdict(row.dm_p_holm)})")

tests_df

## INTERPRETATION

Post-hoc analysis of what drives the forecasts and how it relates to the MRFO selections. None
of it feeds back into the modelling choices above.

### SHAP: dependence plot and comparison against both MRFO selections

SHAP importance, computed from the full-feature XGBoost model, gives an independent
check on which features genuinely matter. The temperature dependence plot shows how the
model uses temperature (e.g. heating/cooling nonlinearity), and the comparison table shows
whether the features each MRFO run dropped were genuinely low-importance, or whether the
search missed something SHAP considers relevant. This either validates or challenges MRFO's
feature drops.

In [ ]:
explainer = shap.TreeExplainer(xgb_results["xgb_full"]["model"])
shap_values = explainer.shap_values(xgb_results["xgb_full"]["X_test"])

shap.summary_plot(shap_values, xgb_results["xgb_full"]["X_test"])

In [ ]:
shap.dependence_plot("temperature", shap_values, xgb_results["xgb_full"]["X_test"])

In [ ]:
shap_importance = pd.Series(np.abs(shap_values).mean(axis=0), index=xgb_results["xgb_full"]["X_test"].columns)
shap_importance = shap_importance.sort_values(ascending=False)

shap_vs_mrfo = pd.DataFrame({
    "shap_importance": shap_importance,
    "in_xgb_mrfo": shap_importance.index.isin(selected_columns_xgb),
    "in_lstm_mrfo": shap_importance.index.isin(selected_columns_lstm),
})
shap_vs_mrfo

#### Feature groups: correlation, grouped SHAP and ablation

Several features carry almost the same information (e.g. `hour`, `hour_sin`, `hour_cos` and
`settlement_period`; or `nd_lag_48` and `nd_rolling_mean_48`). SHAP splits credit between such
features somewhat arbitrarily, so a low per-feature score does not mean the information is
unimportant. Three checks work at the level of feature **groups** instead:

1. A clustered correlation map of the training features, to show which features move together
   and justify the groups.
2. **Grouped SHAP**: SHAP values are additive, so each row's values are summed within a group
   before taking the mean absolute value. The table also shows how many of each group's
   features each MRFO run kept.
3. **Group ablation**: XGBoost (same settings as `xgb_full`) is retrained without each group,
   and the rise in test RMSE shows how much that information matters when the other groups
   can't substitute for it. This is post-hoc analysis of the final model; nothing here feeds
   back into modelling choices.

In [ ]:
FEATURE_GROUPS = {
    "time of day": ["settlement_period", "hour", "hour_sin", "hour_cos"],
    "day of week": ["day_of_week", "day_of_week_sin", "day_of_week_cos"],
    "season / date": ["day_of_month", "day_of_year", "week_of_year", "month", "quarter", "month_sin", "month_cos"],
    "holiday": ["is_holiday"],
    "demand lags": ["nd_lag_48", "nd_lag_288", "nd_lag_336"],
    "demand rolling stats": ["nd_rolling_mean_48", "nd_rolling_std_48", "nd_rolling_mean_336", "nd_rolling_std_336"],
    "weather": ["temperature"],
    "embedded generation": ["embedded_wind_generation", "embedded_solar_generation"],
    "grid operation (lag 48)": [
        "non_bm_stor_lag_48", "pump_storage_pumping_lag_48",
        "ifa_flow_lag_48", "ifa2_flow_lag_48", "britned_flow_lag_48",
        "moyle_flow_lag_48", "east_west_flow_lag_48", "nemo_flow_lag_48",
    ],
}

feature_columns = full.columns.drop("nd")
grouped = [col for group in FEATURE_GROUPS.values() for col in group]
# Every feature must be in exactly one group, so new features can't silently drop out
assert sorted(grouped) == sorted(feature_columns), (
    set(feature_columns) ^ set(grouped)
)

# Clustered Spearman correlation of the training features (a sample, for speed)
X_train_full = train_test_split(full[feature_columns], test_size=0.2, shuffle=False)[0]
corr = X_train_full.sample(20_000, random_state=42).corr(method="spearman")
sns.clustermap(corr, cmap="vlag", vmin=-1, vmax=1, figsize=(12, 12))
plt.show()

In [ ]:
X_test_xgb = xgb_results["xgb_full"]["X_test"]
shap_df = pd.DataFrame(shap_values, columns=X_test_xgb.columns, index=X_test_xgb.index)

group_table = pd.DataFrame({
    group: {
        "grouped_shap": shap_df[cols].sum(axis=1).abs().mean(),
        "n_features": len(cols),
        "kept_by_xgb_mrfo": sum(col in selected_columns_xgb for col in cols),
        "kept_by_lstm_mrfo": sum(col in selected_columns_lstm for col in cols),
    }
    for group, cols in FEATURE_GROUPS.items()
}).T.sort_values("grouped_shap", ascending=False)
group_table

In [ ]:
# Group ablation: retrain XGBoost without each group, compare test RMSE with xgb_full
X_train_ablation, X_test_ablation, y_train_ablation, y_test_ablation = train_test_split(
    full[feature_columns], full["nd"], test_size=0.2, shuffle=False
)
xgb_full_params = xgb_results["xgb_full"]["model"].get_params()
rmse_all_features = xgb_results["xgb_full"]["rmse_test"]

ablation_rmse = {}
for group, cols in FEATURE_GROUPS.items():
    model = xgb.XGBRegressor(**xgb_full_params)
    model.fit(X_train_ablation.drop(columns=cols), y_train_ablation)
    preds = model.predict(X_test_ablation.drop(columns=cols))
    ablation_rmse[group] = evaluate(y_test_ablation, preds, X_test_ablation.index)["rmse_test"]

group_table["rmse_without_group"] = pd.Series(ablation_rmse)
group_table["rmse_increase_pct"] = 100 * (group_table["rmse_without_group"] / rmse_all_features - 1)
group_table.sort_values("rmse_increase_pct", ascending=False)

### Granger causality: does past temperature help predict demand?

This is a supporting analysis, separate from the model comparison. It asks whether *past*
temperature improves the prediction of demand beyond what demand's own history provides.
(The day-ahead models use *same-time* temperature under the perfect-forecast assumption, so
SHAP and the group ablation above answer "how much does temperature matter to the models"
more directly.)

Choices, to make the test meaningful rather than trivially significant:

- **Training portion only**, like MRFO, so no analysis touches the test set.
- **Weekly (336-step) differencing** of both series. This removes the daily and weekly cycles;
  otherwise the test mostly detects that both series follow the time of day.
- **Lag order chosen by AIC** (up to 48 lags = one day) with a VAR model.
- **Reverse-direction sanity check**: demand should not Granger-cause temperature. If it also
  comes out significant, the test is picking up shared seasonality rather than a real effect.
- **Effect size**: with ~170k observations almost any effect is significant, so the out-of-sample
  RMSE reduction from adding lagged temperature to an autoregressive model is also reported.

In [ ]:
# Training portion only; weekly differencing removes the daily and weekly cycles
granger_train = train_test_split(full[["nd", "temperature"]], test_size=0.2, shuffle=False)[0]
granger_data = granger_train.diff(336).dropna()

var_model = VAR(granger_data)
lag_order = max(var_model.select_order(maxlags=48).aic, 1)
print(f"Lag order chosen by AIC: {lag_order}")
var_results = var_model.fit(lag_order)

print(var_results.test_causality("nd", ["temperature"], kind="f").summary())
# Sanity check: should NOT be significant
print(var_results.test_causality("temperature", ["nd"], kind="f").summary())

In [ ]:
# Effect size: out-of-sample RMSE of an autoregressive model of demand, with and without
# lagged temperature (fitted on the first 80% of granger_data, scored on the last 20%)
def lagged_design(frame, columns, n_lags):
    return pd.concat(
        {f"{col}_lag{k}": frame[col].shift(k) for col in columns for k in range(1, n_lags + 1)},
        axis=1,
    )


design_ar = lagged_design(granger_data, ["nd"], lag_order)
design_with_temp = lagged_design(granger_data, ["nd", "temperature"], lag_order)
rows_ok = design_with_temp.notna().all(axis=1)
split = int(0.8 * rows_ok.sum())
target = granger_data["nd"][rows_ok]


def holdout_rmse(design):
    design = design[rows_ok]
    fit = LinearRegression().fit(design[:split], target[:split])
    return np.sqrt(mean_squared_error(target[split:], fit.predict(design[split:])))


rmse_ar, rmse_with_temp = holdout_rmse(design_ar), holdout_rmse(design_with_temp)
print(f"Hold-out RMSE, demand lags only:         {rmse_ar:.1f} MW")
print(f"Hold-out RMSE, plus lagged temperature:  {rmse_with_temp:.1f} MW")
print(f"Reduction: {100 * (1 - rmse_with_temp / rmse_ar):.2f}%")